# API Experimentation
I currently have access to CarAPI's free version, which includes the dataset from 2015-2020 and 100 calls per month. While I am waiting for an academic key and full dataset access, I will experiment here to familiarize myself with the dataset and create a small version of the eventual decision tree

So far, this notebook holds my experiments to identify which endpoints to use, which features to extract from each endpoint, and how to join each endpoint call into one CSV file



In [1]:
import os
from pathlib import Path
import codecarbon.emissions_tracker as et

et.count_physical_cpus = lambda: 1

from codecarbon import EmissionsTracker

LOG_DIR = Path("data/carbon_logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)

tracker = EmissionsTracker(
    project_name="api_experiment",
    output_dir=str(LOG_DIR),
    output_file="api_experiment.csv",
)
tracker.start()

[codecarbon WARNING @ 14:55:49] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon INFO @ 14:55:50] Emissions data (if any) will be saved to file c:\Users\kykin\OneDrive\Desktop\DSAN5550\dsan_5550_project\data\carbon_logs\api_experiment.csv
[codecarbon INFO @ 14:55:50] [setup] RAM Tracking...
[codecarbon INFO @ 14:55:50] [setup] CPU Tracking...
[codecarbon INFO @ 14:55:50] [setup] GPU Tracking...
[codecarbon INFO @ 14:55:50] Tracking CPU via the Windows Energy Meter Interface (RAPL)
[codecarbon INFO @ 14:55:50] Tracking Nvidia GPUs via PyNVML
[codecarbon INFO @ 14:55:51] The below tracking methods have been set up:
                RAM Tracking Method: RAM power estimation model
                CPU Tracking Method: Windows EMI
                GPU Tracking Method: pynvml
            
[codecarbon INFO @ 14:55:51] >>> Tracker's metadata:
[codecarbon INFO @ 14:55:51]   Platform system: Windows-11-10.0.26200-SP0
[codecarbon INFO @ 14:55:51]   Python version: 3.

In [2]:
#packages in my environment:
# used pip install for:
    #pandas, numpy, matplotlib, seaborn, requests
%pip list

Package                 Version
----------------------- -----------
annotated-doc           0.0.5
annotated-types         0.8.0
appnope                 1.0.0
arrow                   1.4.0
asttokens               3.0.2
Authlib                 1.9.5
certifi                 2026.10.23
cffi                    2.1.1
charset-normalizer      3.5.2
click                   8.5.1
codecarbon              3.3.2
colorama                0.4.6
comm                    0.2.3
contourpy               1.4.0
cryptography            50.0.2
cycler                  0.12.1
debugpy                 1.8.22
decorator               5.3.1
exceptiongroup          1.3.1
executing               2.3.0
fonttools               4.68.0
idna                    3.20
ipykernel               6.31.0
ipython                 8.39.0
ipython_pygments_lexers 1.1.1
jedi                    0.20.1
joserfc                 1.8.0
jupyter_client          8.10.0
jupyter_core            5.9.1
kiwisolver              1.5.1
markdown-it-py      

In [3]:
#requesting all 2020 makes from the free tier
import json, requests

r = requests.get("https://carapi.app/api/makes",
                 params={"year": 2020, "limit": 100},
                 headers={"accept": "application/json"},
                 timeout=30)
r.raise_for_status()

with open("data/sample_set/makes_2020.json", "w") as f:
    json.dump(r.json(), f, indent=2)

In [4]:
# turning that into a csv in the dataset
# This CSV only has model names
import pandas as pd

df = pd.DataFrame(r.json()["data"])
df.to_csv("data/sample_set/makes_2020.csv", index=False)

In [5]:
#This call pulls make, model (name), and make_id for the first 100 cars in the dataset. 

import json, time
from pathlib import Path
import pandas as pd
import requests

BASE = "https://carapi.app"
HEADERS = {"accept": "application/json"} 
YEAR = 2020
OUT_DIR = Path("data/sample_set")
OUT_DIR.mkdir(parents=True, exist_ok=True)


def get_json(path, **params):
    while True:
        r = requests.get(f"{BASE}{path}", headers=HEADERS, params=params, timeout=30)
        if r.status_code == 429:
            time.sleep(60)
            continue
        r.raise_for_status()
        return r.json()


def fetch_all(path, conditions=None, **params):
    """Page a collection endpoint. conditions = list of {field, op, val} dicts."""
    if conditions:
        params["json"] = json.dumps(conditions) 
    rows, page = [], 1
    while True:
        body = get_json(path, **{**params, "page": page, "limit": 100})
        batch = body.get("data", [])
        rows.extend(batch)
        if not batch or page >= body.get("pages", 1):
            return rows
        page += 1
        time.sleep(0.2)

models = pd.DataFrame(fetch_all("/api/models/v2",
                                conditions=[{"field": "year", "op": "in", "val": [YEAR]}]))

cols = ["year", "make", "name", "make_id", "make_model_id"]
models = models[[c for c in cols if c in models.columns]]
models.to_csv(OUT_DIR / f"models_{YEAR}.csv", index=False)

print(f"{len(models)} rows, {models['make'].nunique()} makes")
print(models.head())

100 rows, 6 makes
    make name  make_id
0  Acura  ILX        1
1  Acura  MDX        1
2  Acura  NSX        1
3  Acura  RDX        1
4  Acura  RLX        1


In [6]:
import json, requests

r = requests.get("https://carapi.app/api/models/v2",
                 headers={"accept": "application/json"},
                 params={"limit": 1, "page": 1,
                         "json": json.dumps([{"field": "year", "op": "in", "val": [2020]}])},
                 timeout=30)
r.raise_for_status()
body = r.json()

print(body.keys())            # wrapper keys: data, pages, etc.
print(json.dumps(body["data"][0], indent=2))

dict_keys(['collection', 'data'])
{
  "id": 6946,
  "make_id": 1,
  "make": "Acura",
  "name": "ILX"
}


In [7]:
# trims

import json, requests

r = requests.get("https://carapi.app/api/trims/v2",
                 headers={"accept": "application/json"},
                 params={"limit": 1, "page": 1,
                         "json": json.dumps([{"field": "year", "op": "in", "val": [2020]}])},
                 timeout=30)
r.raise_for_status()
body = r.json()

print(body.keys())
print(json.dumps(body["data"], indent=2))

dict_keys(['collection', 'data'])
[
  {
    "id": 6309,
    "make_id": 1,
    "model_id": 5387,
    "submodel_id": 72327,
    "year": 2020,
    "make": "Acura",
    "model": "NSX",
    "series": null,
    "submodel": "Base",
    "trim": "Base",
    "description": "2dr Coupe AWD (3.5L 6cyl Turbo gas/electric hybrid 9AM)",
    "msrp": 157500,
    "invoice": 146251,
    "created": "2023-06-29T20:59:59-04:00",
    "modified": "2026-10-19T22:31:28-04:00"
  }
]


In [8]:
# bodies

import json, requests

r = requests.get("https://carapi.app/api/bodies/v2",
                 headers={"accept": "application/json"},
                 params={"limit": 1, "page": 1,
                         "json": json.dumps([{"field": "year", "op": "in", "val": [2020]}])},
                 timeout=30)
r.raise_for_status()
body = r.json()

print(body.keys())
print(json.dumps(body["data"], indent=2))

dict_keys(['collection', 'data'])
[
  {
    "id": 6309,
    "make_id": 1,
    "model_id": 5387,
    "submodel_id": 72327,
    "trim_id": 6309,
    "year": 2020,
    "make": "Acura",
    "model": "NSX",
    "series": null,
    "submodel": "Base",
    "trim": "Base",
    "trim_description": "2dr Coupe AWD (3.5L 6cyl Turbo gas/electric hybrid 9AM)",
    "type": "Coupe",
    "doors": 2,
    "length": "176.0",
    "width": "73.3",
    "seats": 2,
    "height": "47.8",
    "wheel_base": "103.5",
    "front_track": null,
    "rear_track": null,
    "ground_clearance": "3.7",
    "cargo_capacity": "4.4",
    "max_cargo_capacity": "4.4",
    "curb_weight": 3878,
    "gross_weight": null,
    "max_payload": null,
    "max_towing_capacity": null
  }
]


In [9]:
# engines

import json, requests

r = requests.get("https://carapi.app/api/engines/v2",
                 headers={"accept": "application/json"},
                 params={"limit": 1, "page": 1,
                         "json": json.dumps([{"field": "year", "op": "in", "val": [2020]}])},
                 timeout=30)
r.raise_for_status()
body = r.json()

print(body.keys())
print(json.dumps(body["data"], indent=2))

dict_keys(['collection', 'data'])
[
  {
    "id": 6309,
    "make_id": 1,
    "model_id": 5387,
    "submodel_id": 72327,
    "trim_id": 6309,
    "year": 2020,
    "make": "Acura",
    "model": "NSX",
    "series": null,
    "submodel": "Base",
    "trim": "Base",
    "trim_description": "2dr Coupe AWD (3.5L 6cyl Turbo gas/electric hybrid 9AM)",
    "engine_type": "hybrid",
    "fuel_type": "premium unleaded (required)",
    "cylinders": "V6",
    "size": "3.5",
    "horsepower_hp": 573,
    "horsepower_rpm": 6500,
    "torque_ft_lbs": 476,
    "torque_rpm": 2000,
    "valves": 24,
    "valve_timing": "Variable",
    "cam_type": "Double overhead cam (DOHC)",
    "drive_type": "all wheel drive",
    "transmission": "9-speed automated manual"
  }
]


In [10]:
# mileage

import json, requests

r = requests.get("https://carapi.app/api/mileages/v2",
                 headers={"accept": "application/json"},
                 params={"limit": 1, "page": 1,
                         "json": json.dumps([{"field": "year", "op": "in", "val": [2020]}])},
                 timeout=30)
r.raise_for_status()
body = r.json()

print(body.keys())
print(json.dumps(body["data"], indent=2))

[codecarbon INFO @ 14:56:21] Energy consumed for RAM : 0.000167 kWh. RAM Power : 20.0 W
[codecarbon INFO @ 14:56:21] Energy consumed for all GPUs : 0.000101 kWh. Total GPU Power : 12.819503428020049 W
[codecarbon INFO @ 14:56:21] Delta energy consumed for CPU with windows_emi : 0.000138 kWh, power : 33.95891030708592 W
[codecarbon INFO @ 14:56:21] Energy consumed for All CPU : 0.000280 kWh
[codecarbon INFO @ 14:56:21] 0.000548 kWh of electricity and 0.000000 L of water were used since the beginning.


dict_keys(['collection', 'data'])
[
  {
    "id": 6309,
    "make_id": 1,
    "model_id": 5387,
    "submodel_id": 72327,
    "trim_id": 6309,
    "year": 2020,
    "make": "Acura",
    "model": "NSX",
    "series": null,
    "submodel": "Base",
    "trim": "Base",
    "trim_description": "2dr Coupe AWD (3.5L 6cyl Turbo gas/electric hybrid 9AM)",
    "fuel_tank_capacity": "15.6",
    "combined_mpg": 21,
    "epa_city_mpg": 21,
    "epa_highway_mpg": 22,
    "range_city": 327,
    "range_highway": 343,
    "battery_capacity_electric": null,
    "epa_time_to_charge_hr_240v_electric": null,
    "epa_kwh_100_mi_electric": null,
    "range_electric": null,
    "epa_highway_mpg_electric": null,
    "epa_city_mpg_electric": null,
    "epa_combined_mpg_electric": null
  }
]


The trims, bodies, engines, and mileages endpoints will get me all the information I need.

trims: make_id, model_id, submodel_id, year, make, model, submodel, trim, type, msrp, invoice

bodies: trim_id, type, trim_description

engines: engine_type, fuel_type

mileages: fuel_tank_capacity, combined_mpg, epa_city_mpg, epa_highway_mpg, range_city, range_highway, battery_capacity_electric, epa_time_to_charge_hr_240v_electric, epa_kwh_100_mi_electric, range_electric, epa_highway_mpg_electric, epa_city_mpg_electric, epa_combined_mpg_electric

I want to make a function to call all of that, join on IDs, and create a CSV

In [11]:
# helper functions and setup
import json
import time
from pathlib import Path

import pandas as pd
import requests

#making the paths easier to write
BASE = "https://carapi.app"
HEADERS = {"accept": "application/json"}   # add "Authorization": f"Bearer {JWT}" if I hit 403/when paid
KEYS = ["make_id", "model_id", "submodel_id"]

# request the json objects from API
def get_json(path, **params):
    """Single GET with 429 back-off."""
    while True:
        r = requests.get(f"{BASE}{path}", headers=HEADERS, params=params, timeout=30)
        if r.status_code == 429:
            time.sleep(60)
            continue
        r.raise_for_status()
        return r.json()

# fetch all rows/objects/cars
def fetch_all(path, conditions=None, **params):
    """Page through a collection endpoint; returns every row as a list of dicts."""
    if conditions:
        params["json"] = json.dumps(conditions)
    rows, page = [], 1
    while True:
        body = get_json(path, **{**params, "page": page, "limit": 100})
        batch = body.get("data", [])
        rows.extend(batch)
        coll = body.get("collection", {})
        total_pages = coll.get("pages", coll.get("total_pages", 1))
        if not batch or page >= total_pages:
            return rows
        page += 1
        time.sleep(0.2)

# error control if the schema changes during data collection
def keep(df, cols):
    """Keep requested columns that exist, so a schema change doesn't crash the run."""
    return df[[c for c in cols if c in df.columns]]



In [12]:
def pull_vehicle_data(years=(2020,), n_rows=100, out_csv=None): #takes a tuple of years, rows per year, option to output a CSV
    """Pull trims/bodies/engines/mileages for one or more model years."""
    if isinstance(years, int):
        years = [years]

    frames = []
    for year in years:
        cond = [{"field": "year", "op": "in", "val": [year]}]

        trims = pd.DataFrame(fetch_all("/api/trims/v2", conditions=cond)).head(n_rows)
        trims = keep(trims, KEYS + ["year", "make", "model", "submodel",
                                    "trim", "description", "type", "msrp", "invoice"])
        if trims.empty:
            print(f"no trims for {year} — skipping")
            continue

        combos = trims[KEYS].drop_duplicates()

        def by_keys(endpoint):
            out = []
            for i in range(0, len(combos), 50):
                chunk = combos.iloc[i:i + 50]
                chunk_conds = [{"field": k, "op": "in", "val": chunk[k].dropna().unique().tolist()}
                               for k in KEYS]
                out += fetch_all(endpoint, conditions=chunk_conds)
                time.sleep(0.2)
            return pd.DataFrame(out)

        bodies = keep(by_keys("/api/bodies/v2"),
                      KEYS + ["type"]).rename(columns={"type": "body_type"}) #body type rename
        engines = keep(by_keys("/api/engines/v2"), KEYS + ["engine_type", "fuel_type"])
        mileages = keep(by_keys("/api/mileages/v2"), KEYS + [
            "fuel_tank_capacity", "combined_mpg", "epa_city_mpg", "epa_highway_mpg",
            "range_city", "range_highway", "battery_capacity_electric",
            "epa_time_to_charge_hr_240v_electric", "epa_kwh_100_mi_electric",
            "range_electric", "epa_highway_mpg_electric", "epa_city_mpg_electric",
            "epa_combined_mpg_electric"])

        def collapse(df, how):
            if df.empty:
                return df.drop_duplicates(subset=KEYS)
            return df.groupby(KEYS, as_index=False).agg(how)

        bodies = collapse(bodies, {c: "first" for c in bodies.columns if c not in KEYS})
        engines = collapse(engines, {c: "first" for c in engines.columns if c not in KEYS})
        mileages = collapse(mileages, {c: "max" for c in mileages.columns if c not in KEYS})

        frames.append(trims
                      .merge(bodies,  on=KEYS, how="left")
                      .merge(engines, on=KEYS, how="left")
                      .merge(mileages, on=KEYS, how="left"))

    if not frames:
        raise RuntimeError(f"No data returned for {years} — check auth or the year filter.")

    df = pd.concat(frames, ignore_index=True)

    if out_csv:
        Path(out_csv).parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(out_csv, index=False)

    return df

In [13]:
df = pull_vehicle_data(years=range(2015, 2021), n_rows=10, 
#remember right endpoint is not inclusive
#n = number of rows called for each year
                       out_csv="data/sample_set/vehicles_2015_2020.csv")
print(df.shape)
print(df["year"].value_counts().sort_index())

[codecarbon INFO @ 14:57:06] Energy consumed for RAM : 0.000417 kWh. RAM Power : 20.0 W
[codecarbon INFO @ 14:57:06] Energy consumed for all GPUs : 0.000243 kWh. Total GPU Power : 9.807188270375942 W
[codecarbon INFO @ 14:57:06] Delta energy consumed for CPU with windows_emi : 0.000108 kWh, power : 22.5364850938283 W
[codecarbon INFO @ 14:57:06] Energy consumed for All CPU : 0.000567 kWh
[codecarbon INFO @ 14:57:06] 0.001227 kWh of electricity and 0.000000 L of water were used since the beginning.
[codecarbon INFO @ 14:57:21] Energy consumed for RAM : 0.000500 kWh. RAM Power : 20.0 W
[codecarbon INFO @ 14:57:21] Energy consumed for all GPUs : 0.000279 kWh. Total GPU Power : 8.629421216949384 W
[codecarbon INFO @ 14:57:21] Delta energy consumed for CPU with windows_emi : 0.000083 kWh, power : 25.833414404876134 W
[codecarbon INFO @ 14:57:21] Energy consumed for All CPU : 0.000650 kWh
[codecarbon INFO @ 14:57:21] 0.001429 kWh of electricity and 0.000000 L of water were used since the beg

(60, 27)
year
2015    10
2016    10
2017    10
2018    10
2019    10
2020    10
Name: count, dtype: int64


The sample dataset, which is available at https://carapi.app/features/vehicle-csv-download/ , has roughly 17k rows. This dataset will be loaded as CSVs directly into the project, to avoid expensive API calls. 

The above call for 90 rows total from 2015 to 2020 took _____ minutes to run, and was allowed on the free version. We should expect another 17k+ rows for data from 2021 to 2027.

I attempted to call 100 rows per year, 600 total, and the process took over 20 minutes to complete. I canceled the request at that point and re-ran the smaller request. It may be worth it to get the paid version for speed.

In [14]:
emissions = tracker.stop()
print(f"{emissions:.6f} kg CO2eq")

[codecarbon INFO @ 14:59:33] Energy consumed for RAM : 0.001234 kWh. RAM Power : 20.0 W
[codecarbon INFO @ 14:59:33] Energy consumed for all GPUs : 0.000595 kWh. Total GPU Power : 8.314943574763234 W
[codecarbon INFO @ 14:59:33] Delta energy consumed for CPU with windows_emi : 0.000081 kWh, power : 19.661093464736272 W
[codecarbon INFO @ 14:59:33] Energy consumed for All CPU : 0.001356 kWh
[codecarbon INFO @ 14:59:33] 0.003185 kWh of electricity and 0.000000 L of water were used since the beginning.
[codecarbon INFO @ 14:59:33] 0.003361 g.CO2eq/s mean an estimation of 105.99990920495986 kg.CO2eq/year


0.000780 kg CO2eq
